<a href="https://colab.research.google.com/github/ryanmart25/bird-song-recognizer/blob/utils_setup/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bird Species Identifier
A model for identifying the species of bird from audio.

## Imports

In [ ]:
ITERATION = 0
PAUL = True # paul, you are running in an environment with a different keras backend than us, and are using pytorch instead of tensorflow.
# use this flag to gate code that should be run when only you want it to run. If this feels like a clunky and bad idea, feel free to disregard this.
# In general, my idea for these flags was they could be used to section off highly experimental / broken code, or code that only works in a specific
# environment that others might not have.
RYAN = True
BEN = True
WINDOW_SIZE = 7
OPTIMIZER_LEARNING_RATE = 0.001
GLOBAL_DISABLE = True; #When set to true, disables unnecessary code. Leave as is unless you neeD



In [14]:
import pandas as pd
import numpy as np
from pathlib import Path
import os
import sys
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, auc, mean_squared_error
import matplotlib.pyplot as plt
from collections.abc import Sequence
from sklearn import preprocessing
%matplotlib inline
import csv
import glob
from IPython.display import Image
import seaborn as sns

import os

if (PAUL):
    os.environ["KERAS_BACKEND"] = "torch"   # must be set before 'import keras'
    import torch
    import keras
    from keras.utils import plot_model, load_img, img_to_array
    from keras import layers, models
    from keras.models import Sequential, Model
    from keras.layers import Dense, Dropout, Input, Flatten, Conv2D, MaxPooling2D, concatenate, Conv1D, MaxPooling1D, Dropout, Input, Conv2D, MaxPooling2D
    from keras.utils import plot_model
    from keras.optimizers import Adam
    from keras.callbacks import EarlyStopping, ModelCheckpoint
    print("Torch CUDA available:", torch.cuda.is_available())  # Should be true when running GPU processing, else ignore

else:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers, models
    from tensorflow.keras.models import Sequential, Model
    from tensorflow.keras.layers import Dense, Dropout, Input, Flatten, Conv2D, MaxPooling2D, concatenate, Conv1D, MaxPooling1D
    from tensorflow.keras.utils import plot_model
    from tensorflow.keras.optimizers import Adam
    from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
    from tensorflow.keras.utils import plot_model, load_img, img_to_array



Torch CUDA available: True


## Global Control Flow Flags and Program Configuration

## Define Helper Methods

In [28]:
def plot_losses(history, base_path, iteration:int):
    # Plot training & validation loss over epochs
    plt.plot(history.history["loss"], label="Training Loss")
    plt.plot(history.history["val_loss"], label="Validation Loss")
    plt.ylim(bottom=0.0, top=10.0)
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training vs. Validation Loss")
    plt.legend()
    plt.savefig(
        os.path.join(base_path, f"training-validiation-loss--epoch---Model {iteration}")
    )
    plt.close()


def print_schema(dataframe: pd.DataFrame):
    print('~~~~~~dataframe schema~~~~~~')
    print(f"Dataframe shape: {dataframe.shape} | Dataframe length: {len(dataframe)}")
    print('Column labels: ')
    print(dataframe.columns)
    print('Dataframe head: ')
    print(f"{dataframe.head()}")
def print_column(dataframe: pd.DataFrame, columns: str | list[str]):
    if isinstance(columns, list):
        for i, label in enumerate(columns):
            print(f"column {i}")
            print(dataframe[label])
    else:
        print(dataframe[columns])
# Encode text values to dummy variables(i.e. [1,0,0],[0,1,0],[0,0,1] for red,green,blue)
def encode_text_dummy(df, name):
    dummies = pd.get_dummies(df[name])
    for x in dummies.columns:
        dummy_name = "{}-{}".format(name, x)
        df[dummy_name] = dummies[x]
    df.drop(name, axis=1, inplace=True)


# Encode text values to indexes(i.e. [1],[2],[3] for red,green,blue).
def encode_text_index(df, name):
    le = preprocessing.LabelEncoder()
    df[name] = le.fit_transform(df[name])
    return le.classes_


# Encode a numeric column as zscores
def encode_numeric_zscore(df, name, mean=None, sd=None):
    if mean is None:
        mean = df[name].mean()

    if sd is None:
        sd = df[name].std()

    df[name] = (df[name] - mean) / sd


# Convert all missing values in the specified column to the median
def missing_median(df, name):
    med = df[name].median()
    df[name] = df[name].fillna(med)


# Convert all missing values in the specified column to the default
def missing_default(df, name, default_value):
    df[name] = df[name].fillna(default_value)


# Convert a Pandas dataframe to the x,y inputs that TensorFlow needs
def to_xy(df, target):
    result = []
    for x in df.columns:
        if x != target:
            result.append(x)
    # find out the type of the target column.
    target_type = df[target].dtypes
    target_type = target_type[0] if isinstance(target_type, Sequence) else target_type
    # Encode to int for classification, float otherwise. TensorFlow likes 32 bits.
    #if target_type in (np.int64, np.int32):
        ## Classification
        #dummies = pd.get_dummies(df[target])
        #return df[result].values.astype(np.float32), dummies.values.astype(np.float32)
    #else#:
        ## Regression
    return df[result].values.astype(np.float32), df[target].values.astype(np.float32)

# Nicely formatted time string
def hms_string(sec_elapsed):
    h = int(sec_elapsed / (60 * 60))
    m = int((sec_elapsed % (60 * 60)) / 60)
    s = sec_elapsed % 60
    return "{}:{:>02}:{:>05.2f}".format(h, m, s)


# Regression chart.
def chart_regression(path, pred,y,sort=True):
    t = pd.DataFrame({'pred' : pred, 'y' : y.flatten()})
    if sort:
        t.sort_values(by=['y'],inplace=True)
    b = plt.plot(t['pred'].tolist(),label='prediction')
    a = plt.plot(t['y'].tolist(),label='expected')

    plt.ylabel('output')
    plt.legend()
    plt.savefig(path)
    plt.close()

# Remove all rows where the specified column is +/- sd standard deviations
def remove_outliers(df, name, sd):
    drop_rows = df.index[(np.abs(df[name] - df[name].mean()) >= (sd * df[name].std()))]
    df.drop(drop_rows, axis=0, inplace=True)


# Encode a column to a range between normalized_low and normalized_high.
def encode_numeric_range(df, name, normalized_low=-1, normalized_high=1,
                         data_low=None, data_high=None):
    if data_low is None:
        data_low = min(df[name])
        data_high = max(df[name])

    df[name] = ((df[name] - data_low) / (data_high - data_low)) \
               * (normalized_high - normalized_low) + normalized_low



## Import and Read Datasets

In [41]:
# import an API key for the dataset
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"ryanmartinez2025","key":"997d4c5c308e99f7d25bee0d94d11138"}'}

In [49]:
if (not GLOBAL_DISABLE):
    print("didn't run")

In [ ]:
if (not GLOBAL_DISABLE):
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json



## Download Dataset

In [ ]:
if (not GLOBAL_DISABLE):
    # The following code will only execute
    # successfully when compression is complete

    import kagglehub

    # Download latest version
    path = kagglehub.dataset_download("ryanmartinez2025/birdcall-spectrograms")

    print("Path to dataset files:", path)

100%|██████████| 53.2M/53.2M [00:03<00:00, 15.4MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/ryanmartinez2025/bird-calls-spectrograms/versions/1


## Configure Environment

In [ ]:
import os
import sys
output_path = os.path.join(os.getcwd(), "output")
iteration_path= os.path.join(output_path, f"iteration-{ITERATION}")
base_dataset_path = "./" #changed this from user-dependent path naming
# idk how you are going to seperate train/test/val. Are you seperating them into
# seperate .csv files or are we simply going to do 1 big csv file that we use
# train_test_split on?
train_ds_path = os.path.join(base_dataset_path, "train")
test_ds_path = os.path.join(base_dataset_path, "test")
val_ds_path = os.path.join(base_dataset_path, "val")
try:
  os.makedirs(iteration_path)
except FileExistsError as e:
  print(f"{iteration_path} already exists. exiting to save previous work.")
  sys.exit(0)


## Code for generating CSV from raw data

In [ ]:
if (not GLOBAL_DISABLE):

  import os
  import csv
  from pathlib import Path
  import pandas as pd
  CWD = os.getcwd()
  test_path_relative= "data/spectrogram_dataset/test"
  testPath = os.path.join(CWD, test_path_relative)
  train_path_relative= "data/spectrogram_dataset/train"
  trainPath = os.path.join(CWD, train_path_relative)


  #create csv 
  #for every bird folder
    #for every file
      #log file path (relative) into column and corresponding row
      #log corresponding bird species (dashed lines for spaces) based off file path

  def create_bird_csv(path, csvName):
    
    
    rows = []

    j=0 
    for root,_,filenames in os.walk(path):
      base_path = Path.cwd()
      root_path = Path(root)
      rel_parts = root_path.relative_to(base_path).parts
      species_name = root_path.name

      species = rel_parts[0]
      species_slug = species.replace(" ", "-")
      # print(root)
      # print(filenames)
      i = 0
      for filename in filenames:
        file_path = Path(root_path) / filename
        rel_path = file_path.relative_to(base_path) 


        # print(os.path.isfile(os.path.join(root, filename)))

        new = root_path / f"{species_slug}-{i}.png"
        rel_to_cwd = new.resolve().relative_to(Path.cwd())
        rows.append({"Filepath": str(rel_path), "Bird-Species": str(species_name) })
        i = i + 1
        j = j + 1

    df = pd.DataFrame(rows, columns= [
      "Filepath",
      "Bird-Species"
    ])
    df.to_csv(csvName, index=False)



  create_bird_csv(testPath,"testCSV.csv")
  create_bird_csv(trainPath, "trainCSV.csv")

## Creating Dataframe
- Translates CSV data into useable dataframe for target
- loads all the images based of csv stored file paths
    - Places these into a numpy array 
- one-hot encodes all the text associated with the bird-species

In [ ]:
from PIL import UnidentifiedImageError
df1 = pd.read_csv("trainCSV.csv")
df2 = pd.read_csv("testCSV.csv")

modelDF = pd.concat([df1, df2], ignore_index=True) # created dataframe containing all the target categories and images

print(df1.shape)
print(df2.shape)
print(modelDF.shape)

#forloop here
img_paths = modelDF["Filepath"]
labels_df = modelDF[["Bird-Species"]].copy()

inputImages = []
for path in img_paths:
    try:
        tmpImg = load_img(("./" + path), target_size=(400, 1000))

        img_array = img_to_array(tmpImg)
        img_array = img_array /255.0 #float conversion for cnn
        inputImages.append(img_array)

    except UnidentifiedImageError:
        print(f"Could not identify image file: {path}")
        bad_files.append(path)
    except Exception as e:
        print(f"Error loading {path}: {e}")
        bad_files.append(path)


x = np.stack(inputImages)
encode_text_dummy(labels_df, "Bird-Species")
y = labels_df.to_numpy() #converting to numpy for supposed better performance with pyTorch





(4365, 2)
(687, 2)
(5052, 2)


In [35]:
print(y)

None


## Train/Test Split



In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42) 

In [43]:
full_path = iteration_path #path to current iteration directory to save weights
checkpointer = ModelCheckpoint(filepath=os.path.join(full_path, "best_weights.keras"), verbose=0, save_best_only=True) #save best model
optimizer = Adam(learning_rate = 0.0001)
input_shape = (400,1000,3) #input shape H,W,Channels
num_classes = 41 #num_classes = classes to predict

inputs = layers.Input(shape=input_shape)
#CNN
#block 1
x = layers.Conv2D(32, kernel_size=3, activation='relu', padding='same')(inputs)
x = layers.MaxPooling2D((2, 2))(x)
x = layers.Dropout(0.25)(x)

#block 2
x = layers.Conv2D(64, kernel_size=3, activation='relu', padding='same')(x)
x = layers.MaxPooling2D((2, 2))(x)
x = layers.Dropout(0.25)(x)

#block 3
x = layers.Conv2D(128, kernel_size=3, activation='relu', padding='same')(x)
x = layers.MaxPooling2D((2, 2))(x)
x = layers.Dropout(0.25)(x)

#block 4
x = layers.Conv2D(256, kernel_size=3, activation='relu', padding='same')(x)
x = layers.MaxPooling2D((2, 2))(x)
x = layers.Dropout(0.25)(x)

#reshape for the LSTM
# -> (time, features)
shape = x.shape
x = layers.Reshape((shape[2], shape[1] * shape[3]))(x)

#LSTM for time modeling
x = layers.LSTM(128, return_sequences=True)(x) #output (62,128)
x = layers.Dropout(0.3)(x)
x = layers.LSTM(64, return_sequences=False)(x) #output (128)
x = layers.Dropout(0.3)(x)

#dense layers for classification
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.4)(x)
x = layers.Dense(64, activation='relu')(x)
x = layers.Dropout(0.4)(x)

#output layer (variable number of classes to predict)
outputs = layers.Dense(num_classes, activation='softmax')(x)

#create the model
model = Model(inputs=inputs, outputs=outputs)

#model compilation
model.compile(optimizer=optimizer, loss='categorical_crossentropy')

monitor = EarlyStopping(monitor='val_loss', min_delta=1e-3, patience=5, verbose=2, mode='min', restore_best_weights=True)

history = model.fit(x_train, y_train, validation_data=(x_test, y_test), epochs=100, batch_size=16, callbacks=[checkpointer, monitor])


Epoch 1/100
253/253 ━━━━━━━━━━━━━━━━━━━━ 113s 444ms/step - loss: 3.6179 - val_loss: 3.5124
Epoch 2/100
253/253 ━━━━━━━━━━━━━━━━━━━━ 111s 439ms/step - loss: 3.4985 - val_loss: 3.5012
Epoch 3/100
253/253 ━━━━━━━━━━━━━━━━━━━━ 114s 451ms/step - loss: 3.4354 - val_loss: 3.4164
Epoch 4/100
253/253 ━━━━━━━━━━━━━━━━━━━━ 113s 448ms/step - loss: 3.4035 - val_loss: 3.4183
Epoch 5/100
253/253 ━━━━━━━━━━━━━━━━━━━━ 113s 447ms/step - loss: 3.3947 - val_loss: 3.3764
Epoch 6/100
253/253 ━━━━━━━━━━━━━━━━━━━━ 111s 438ms/step - loss: 3.3524 - val_loss: 3.3667
Epoch 7/100
253/253 ━━━━━━━━━━━━━━━━━━━━ 112s 444ms/step - loss: 3.3104 - val_loss: 3.2367
Epoch 8/100
253/253 ━━━━━━━━━━━━━━━━━━━━ 113s 448ms/step - loss: 3.2830 - val_loss: 3.2289
Epoch 9/100
253/253 ━━━━━━━━━━━━━━━━━━━━ 112s 445ms/step - loss: 3.2399 - val_loss: 3.1207
Epoch 10/100
253/253 ━━━━━━━━━━━━━━━━━━━━ 113s 447ms/step - loss: 3.2010 - val_loss: 3.1297
Epoch 11/100
253/253 ━━━━━━━━━━━━━━━━━━━━ 112s 443ms/step - loss: 3.1570 - val_loss: 3.01

In [ ]:
DEBUG = True
metrics_path = "metrics.txt"
metrics_base = iteration_path + "/"
def redirect(out): # redirect model summary to metrics
    with open(os.path.join(metrics_base, metrics_path), 'a') as file:
        print(out, file=file)
# make prediction and evaluate
model.load_weights(os.path.join(iteration_path, "best-weights.keras"))
prediction = model.predict(x_test)
score = np.sqrt(mean_squared_error(y_test, prediction))
if DEBUG:
    print("Score (RMSE): {}".format(score))
# Write Metrics to file
with open(os.path.join(metrics_base, metrics_path), "x") as file:
    file.write(f"Score (RMSE): {score}\n")
model.summary(print_fn=redirect)
chart_regression(os.path.join(metrics_base, "Lift-Chart"), prediction.flatten(), y_test, sort=True)

FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\snowt\\OneDrive\\Documents\\CSC-180\\finalProject\\bird-song-recognizer\\output\\iteration-0\\best-weights.keras'